# Assignment 05: Clean a Lab Export

`README.md` gives each task's steps and checkpoints; the headings here match its numbering. Run each cell with `Shift+Enter`. When you finish, click **Restart**, then **Run All**, and run `python3 check_assignment.py` in the terminal.

In [12]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

DATA_PATH = Path("data") / "lab_visits.csv"
if not DATA_PATH.exists():
    subprocess.run([sys.executable, "generate_data.py"], check=True)
manifest = json.loads(Path("data_manifest.json").read_text())
for name, expected in manifest["files"].items():
    digest = hashlib.sha256(Path(name).read_bytes()).hexdigest()
    assert digest == expected["sha256"], f"{name} differs from the supplied data: delete data/ and run this cell again"
    print(name, "matches data_manifest.json")

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
MISSING_PATH = OUTPUT_DIR / "missing_report.csv"
PROFILE_PATH = OUTPUT_DIR / "typed_profile.csv"
CATEGORY_PATH = OUTPUT_DIR / "category_counts.csv"
VALIDATION_PATH = OUTPUT_DIR / "validation_report.csv"
ROW_COUNTS_PATH = OUTPUT_DIR / "row_counts.csv"
SUMMARY_PATH = OUTPUT_DIR / "clean_summary.csv"

NumPy: 2.3.3
pandas: 3.0.5
data/lab_visits.csv matches data_manifest.json


## Task 1: Find missing values and repeats

### 1.1 Count missing values

Run this cell as it is: it reads the export with no options.

In [13]:
labs = pd.read_csv(DATA_PATH)
rows_read = len(labs)
print("shape:", labs.shape)
print(labs.dtypes)
labs.head()



shape: (60918, 8)
patient_id           str
visit_date           str
clinic               str
gender               str
age                  str
weight_kg        float64
hba1c_pct        float64
glucose_mg_dl    float64
dtype: object


,patient_id,visit_date,clinic,gender,age,weight_kg,hba1c_pct,glucose_mg_dl
0,D10425,2024-01-02,Bayview,M,64,122.7,6.7,164.0
1,D10810,2024-01-02,Mission,F,74,42.0,7.7,172.0
2,D11509,2024-01-02,SUNSET,F,57,70.0,7.0,178.0
3,D13213,2024-01-02,Sunset,F,51,62.3,7.1,172.0
4,D13220,2024-01-02,Bayview,male,61,125.4,5.6,104.0


> **Checkpoint: `output/missing_report.csv`**
> Eight rows under the header `column,missing_count,missing_pct`.

In [14]:
missing_report = pd.DataFrame({
    "missing_count": labs.isna().sum(),
    "missing_pct": (labs.isna().mean() * 100).round(1),
})

missing_report.index.name = "column"

missing_report
missing_report.to_csv(MISSING_PATH)

### 1.2 Drop no-lab visits and exact repeats

In [15]:
labs = labs.dropna(
    subset=["hba1c_pct", "glucose_mg_dl"],
    how="all"
)

rows_with_labs = len(labs)

print("rows with labs:", rows_with_labs)
exact_repeats = labs.duplicated().sum()

print("exact repeats:", exact_repeats)
labs = labs.drop_duplicates()

rows_unique = len(labs)

print("unique rows:", rows_unique)

rows with labs: 59977
exact repeats: 491
unique rows: 59486


## Task 2: Replace codes and convert types

> **Checkpoint: `output/typed_profile.csv`**
> Five rows under the header `column,dtype,missing_count,min,max,mean`.

In [16]:
labs = labs.replace({
    "weight_kg": {-999: np.nan},
    "glucose_mg_dl": {-999: np.nan},
})
labs["age"] = pd.to_numeric(
    labs["age"],
    errors="coerce"
).astype("Int64")
labs["visit_date"] = pd.to_datetime(
    labs["visit_date"],
    format="%Y-%m-%d",
    errors="coerce"
)
typed = labs[
    ["age", "glucose_mg_dl", "hba1c_pct", "visit_date", "weight_kg"]
]
typed_profile = pd.DataFrame({
    "dtype": typed.dtypes.astype(str),
    "missing_count": typed.isna().sum(),
    "min": typed.min(),
    "max": typed.max(),
    "mean": typed.mean(numeric_only=True).round(2),
})

typed_profile.index.name = "column"

typed_profile
typed_profile.to_csv(PROFILE_PATH)


## Task 3: Standardize text and encode categories

### 3.1 Clean the text columns

In [17]:
display(labs["clinic"].value_counts(dropna=False))

labs["patient_id"] = labs["patient_id"].str.strip().str.upper()

labs["clinic"] = (
    labs["clinic"]
    .str.strip()
    .str.title()
    .fillna("Unknown")
)

gender_codes = {
    "f": "F",
    "female": "F",
    "m": "M",
    "male": "M",
    "x": "X",
    "nb": "X",
    "nonbinary": "X",
}

labs["gender"] = (
    labs["gender"]
    .str.strip()
    .str.lower()
    .map(gender_codes)
)

clinic
Mission       18539
Sunset        13976
Excelsior     11739
Bayview        7846
mission        1011
sunset          831
excelsior       681
MISSION         626
SUNSET          445
EXCELSIOR       437
bayview         431
 Mission        417
Mission         410
 Sunset         348
Sunset          337
 Excelsior      290
Excelsior       265
BAYVIEW         260
NaN             238
 Bayview        188
Bayview         170
mission           1
Name: count, dtype: int64

### 3.2 Bin HbA1c and count the categories

> **Checkpoint: `output/category_counts.csv`**
> Eleven rows under the header `indicator,visits`, from `clinic_Bayview` to `a1c_category_diabetes`.

In [18]:
labs["a1c_category"] = pd.cut(
    labs["hba1c_pct"],
    bins=[0, 5.6, 6.4, 20],
    labels=["normal", "prediabetes", "diabetes"]
)

coded = pd.get_dummies(
    labs[["clinic", "gender", "a1c_category"]],
    dtype="int64"
)

category_counts = coded.sum()
category_counts.index.name = "indicator"
category_counts.name = "visits"

category_counts
category_counts.to_csv(CATEGORY_PATH)

## Task 4: Flag outliers and count rule failures

Audit flags identify values to review. Missing gender is flagged but allowed in the cleaned table.

> **Checkpoint: `output/validation_report.csv`**
> Six rows under the header `rule,failing_rows`.

In [19]:
q1 = labs["glucose_mg_dl"].quantile(0.25)
q3 = labs["glucose_mg_dl"].quantile(0.75)

iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print("Q1", q1, "Q3", q3, "fences", lower, "to", upper)
labs["glucose_outlier"] = (
    (labs["glucose_mg_dl"] < lower)
    | (labs["glucose_mg_dl"] > upper)
)

print("glucose outliers:", labs["glucose_outlier"].sum())
rules = pd.DataFrame({
    "patient_id_format": labs["patient_id"].str.fullmatch(r"D[0-9]{5}", na=False),
    "visit_date_present": labs["visit_date"].notna(),
    "gender_valid": labs["gender"].isin(["F", "M", "X"]),
    "hba1c_in_range": labs["hba1c_pct"].between(4, 20) | labs["hba1c_pct"].isna(),
    "weight_in_range": labs["weight_kg"].between(30, 250) | labs["weight_kg"].isna(),
    "glucose_within_fences": ~labs["glucose_outlier"],
})

validation_report = (~rules).sum()
validation_report.index.name = "rule"
validation_report.name = "failing_rows"

validation_report
validation_report.to_csv(VALIDATION_PATH)

Q1 119.0 Q3 166.0 fences 48.5 to 236.5
glucose outliers: 859


## Task 5: Clean and summarize

### 5.1 Apply the decisions and count the rows

> **Checkpoint: `output/row_counts.csv`**
> Six rows under the header `step,rows`, from `rows_read` to `weight_blanked`.

In [20]:
clean = labs.copy()

clean = clean.loc[
    rules["patient_id_format"]
    & rules["visit_date_present"]
]
hba1c_outside = (
    (clean["hba1c_pct"] < 4)
    | (clean["hba1c_pct"] > 20)
).sum()

weight_outside = (
    (clean["weight_kg"] < 30)
    | (clean["weight_kg"] > 250)
).sum()

print("hba1c outside:", hba1c_outside)
print("weight outside:", weight_outside)
clean["hba1c_pct"] = clean["hba1c_pct"].where(
    clean["hba1c_pct"].between(4, 20)
)

clean["weight_kg"] = clean["weight_kg"].where(
    clean["weight_kg"].between(30, 250)
)

clean["a1c_category"] = pd.cut(
    clean["hba1c_pct"],
    bins=[0, 5.6, 6.4, 20],
    labels=["normal", "prediabetes", "diabetes"]
)

clean = clean.reset_index(drop=True)
row_counts = pd.Series({
    "rows_read": rows_read,
    "rows_with_labs": rows_with_labs,
    "rows_unique": rows_unique,
    "visits_dropped": len(labs) - len(clean),
    "hba1c_blanked": hba1c_outside,
    "weight_blanked": weight_outside,
}, name="rows")

row_counts.index.name = "step"

row_counts
row_counts.to_csv(ROW_COUNTS_PATH)

hba1c outside: 61
weight outside: 46


### 5.2 Validate, then save the summary

> **Checkpoint: `output/clean_summary.csv`**
> Four rows under the header `,count,mean,std,min,25%,50%,75%,max`.

In [21]:
checks = pd.Series({
    "one row per visit": clean.duplicated(
        subset=["patient_id", "visit_date"]
    ).sum() == 0,
    "patient IDs valid": clean["patient_id"].str.fullmatch(
        r"D[0-9]{5}",
        na=False
    ).all(),
    "visit dates present": clean["visit_date"].notna().all(),
    "hba1c in range": (
        clean["hba1c_pct"].between(4, 20)
        | clean["hba1c_pct"].isna()
    ).all(),
    "weight in range": (
        clean["weight_kg"].between(30, 250)
        | clean["weight_kg"].isna()
    ).all(),
})

print(checks)

assert checks.all(), checks[~checks]
clean_summary = clean[
    ["age", "weight_kg", "hba1c_pct", "glucose_mg_dl"]
].describe().round(2).T

clean_summary
clean_summary.to_csv(SUMMARY_PATH)

one row per visit      True
patient IDs valid      True
visit dates present    True
hba1c in range         True
weight in range        True
dtype: bool


### 5.3 Save the cleaned table as Parquet

> **Checkpoint: `data/labs_clean.parquet`**
> Not committed or checked: `data/` stays out of Git.

In [23]:
parquet_path = Path("data/labs_clean.parquet")

clean.to_parquet(parquet_path)

clean_back = pd.read_parquet(parquet_path)

print(clean_back.dtypes)
print("same as clean:", clean_back.equals(clean))

patient_id                    str
visit_date         datetime64[us]
clinic                        str
gender                        str
age                         Int64
weight_kg                 float64
hba1c_pct                 float64
glucose_mg_dl             float64
a1c_category             category
glucose_outlier              bool
dtype: object
same as clean: True


## Fresh-run check

Click **Restart**, then **Run All**. The last cell prints your score and what to fix.

In [ ]:
from check_assignment import run_checks

_ = run_checks()

Checks: latest from the course repository (main).
[PASS]  4/4  missing report: one row per column
[PASS]  5/5  missing report: missing_count
[PASS]  4/4  missing report: missing_pct
[PASS]  4/4  typed profile: -999 read as missing
[PASS]  4/4  typed profile: age as numbers
[PASS]  3/3  typed profile: unreadable ages left blank
[PASS]  3/3  typed profile: visit dates parsed
[PASS]  2/2  typed profile: undated visits kept
[PASS]  3/3  typed profile: other values unchanged
[PASS]  4/4  category counts: clinic names cleaned
[PASS]  3/3  category counts: missing clinic filled
[PASS]  4/4  category counts: gender codes mapped
[PASS]  5/5  category counts: a1c_category
[PASS]  3/3  category counts: indicator columns
[PASS]  3/3  validation report: one row per rule
[PASS]  2/2  validation report: rule names
[PASS]  4/4  validation report: patient IDs cleaned
[PASS]  5/5  validation report: failing rows
[PASS]  3/3  validation report: glucose outliers flagged
[PASS]  4/4  row counts: no-lab vis